In [ ]:
%pip install -q duckdb numpy numba matplotlib

In [ ]:
import os
import sys

REPO_URL = "https://github.com/payamdav/pycrypto.git"
REPO_NAME = "pycrypto"
BRANCH_NAME = "copilot/create-notebook-data-load-indicators"

if not os.path.exists(REPO_NAME):
    !git clone -b {BRANCH_NAME} {REPO_URL}

REPO_PATH = os.path.abspath(REPO_NAME)
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

import time
import numpy as np
import matplotlib.pyplot as plt
from packages.candle_loader import load_candles
from packages.indicators import rolling_robust_z_score, ma

In [ ]:
ASSET     = "btcusdt"
DATE_FROM = "2024-01-01 00:00:00"
DATE_TO   = "2026-01-01 00:00:00"

In [ ]:
candles = load_candles(ASSET, DATE_FROM, DATE_TO, columns=["vwap", "v", "vb", "vs"])
# columns: ts=0, vwap=1, v=2, vb=3, vs=4

In [ ]:
ZSCORE_WINDOW = 7 * 24 * 60
t0 = time.perf_counter()
v_zscore = rolling_robust_z_score(candles[:, 2], window=ZSCORE_WINDOW)
elapsed = time.perf_counter() - t0
print(f"rolling_robust_z_score elapsed: {elapsed:.4f}s  shape: {v_zscore.shape}")

In [ ]:
MA_WINDOW = 60
t0 = time.perf_counter()
v_ma = ma(candles[:, 2], window=MA_WINDOW)
elapsed = time.perf_counter() - t0
print(f"ma elapsed: {elapsed:.4f}s  shape: {v_ma.shape}")

In [ ]:
x = np.arange(len(candles))
volume = candles[:, 2]

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

axes[0].plot(x, volume, linewidth=0.5)
axes[0].set_ylabel("Volume")
axes[0].set_title("Volume")

axes[1].plot(x, v_ma, linewidth=0.5, color="orange")
axes[1].set_ylabel("MA(Volume, 60)")
axes[1].set_title("MA of Volume (window=60)")

axes[2].plot(x, v_zscore, linewidth=0.5, color="green")
axes[2].set_ylabel("Robust Z-Score")
axes[2].set_title(f"Rolling Robust Z-Score of Volume (window={ZSCORE_WINDOW})")
axes[2].set_xlabel("Candle index")

plt.tight_layout()
plt.show()